# EXIST 2026 — Sexism Meme Classification (Multimodal QLoRA)
**Task:** Multi-label classification of memes using image + text  
**Model:** Qwen2.5-VL-72B-Instruct  
**Method:** QLoRA (4-bit) + PEFT LoRA  
**Hardware:** 2× NVIDIA A40 (48 GB each → 96 GB total)
**Espacio** 400 gb

## 1. Unzip Dataset & Install Dependencies

In [ ]:
ls -lh /workspace/exit-2026-unam.zip
#checa cuanto lleva instalado del zip del dataset

In [ ]:
import shutil, os, subprocess

# Borrar caché de HuggingFace en /root
for path in ["/root/.cache", "/root/hf_cache"]:
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"Borrado: {path}")

# Ver resultado
r = subprocess.run(["df", "-h", "/"], capture_output=True, text=True)
print(r.stdout)

#Verifica cuanto tienes de espacio, se necesitan al menos 200 GB libres para el proceso de fine-tuning

Borrado: /root/.cache
Filesystem      Size  Used Avail Use% Mounted on
overlay         400G  106G  295G  27% /



In [ ]:
import os, zipfile, shutil

ZIP_PATH    = "/workspace/exit-2026-unam.zip"
EXTRACT_DIR = "/workspace/exist2026"

# ── Verificar zip antes de extraer ───────────────────────────────────────────
zip_size = os.path.getsize(ZIP_PATH) / 1e6
print(f"ZIP size : {zip_size:.1f} MB")
if zip_size < 400:
    print("⚠️  El zip parece incompleto (debería pesar ~500 MB+). Verifica la subida.")

# ── Espacio disponible ────────────────────────────────────────────────────────
_, _, free = shutil.disk_usage("/workspace")
print(f"Espacio libre: {free/1e9:.0f} GB")

# ── Extraer ───────────────────────────────────────────────────────────────────
if not os.path.exists(EXTRACT_DIR):
    print(f"\nExtracting {ZIP_PATH} → {EXTRACT_DIR} ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print("Done.")
else:
    print(f"\nAlready extracted: {EXTRACT_DIR}")

# ── Preview estructura ────────────────────────────────────────────────────────
for root, dirs, files in os.walk(EXTRACT_DIR):
    depth = root.replace(EXTRACT_DIR, '').count(os.sep)
    indent = '  ' * depth
    print(f"{indent}{os.path.basename(root)}/")
    if depth >= 2:
        dirs[:] = []
        continue
    sub = '  ' * (depth + 1)
    for f in files[:5]:
        print(f"{sub}{f}")
    if len(files) > 5:
        print(f"{sub}... ({len(files)} files total)")
        
    #extrae zip esto se puede saltar si ya está eso, solo cambia las rutas

ZIP size : 549.6 MB
Espacio libre: 423 GB

Already extracted: /workspace/exist2026
exist2026/
  archive/
    EXIST 2026 Dataset V0.1/


In [1]:
import subprocess, sys

# ── Paso 1: dependencias base de torch (con índice cu124) ─────────────────────
base = [
    "typing_extensions>=4.12.0",
    "torch>=2.6.0",
    "torchvision",
]
for pkg in base:
    print(f"Installing {pkg}...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
         "--index-url", "https://download.pytorch.org/whl/cu124", pkg],
        capture_output=True, text=True
    )
    print(f"  {'OK' if result.returncode == 0 else 'FAILED: ' + result.stderr.strip()}")

# ── Paso 2: resto de paquetes ─────────────────────────────────────────────────
pkgs = [
    "tqdm",
    "pillow",
    "transformers>=4.52.0,<5.0.0",
    "peft>=0.14.0",
    "bitsandbytes>=0.45.0",
    "accelerate>=1.2.1",
    "datasets>=3.2.0",
    "scikit-learn>=1.5.2",
    "iterative-stratification",
    "qwen-vl-utils",
    "matplotlib",
    "huggingface_hub==0.34.0",
    "flash-attn",
]
for pkg in pkgs:
    print(f"Installing {pkg}...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg],
        capture_output=True, text=True
    )
    print(f"  {'OK' if result.returncode == 0 else 'FAILED: ' + result.stderr.strip()}")

# ── Paso 3: quitar hf-xet ────────────────────────────────────────────────────
print("Removing hf-xet...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "hf-xet"],
               capture_output=True)
print("  OK")

print("\nAll packages done.")

Installing typing_extensions>=4.12.0...
  OK
Installing torch>=2.6.0...
  OK
Installing torchvision...
  OK
Installing tqdm...
  OK
Installing pillow...
  OK
Installing transformers>=4.52.0,<5.0.0...
  OK
Installing peft>=0.14.0...
  OK
Installing bitsandbytes>=0.45.0...
  OK
Installing accelerate>=1.2.1...
  OK
Installing datasets>=3.2.0...
  OK
Installing scikit-learn>=1.5.2...
  OK
Installing iterative-stratification...
  OK
Installing qwen-vl-utils...
  OK
Installing matplotlib...
  OK
Installing huggingface_hub==0.34.0...
  OK
Installing flash-attn...
  OK
Removing hf-xet...
  OK

All packages done.


## 2. Imports & Config

In [ ]:
import os, json, re, random, tempfile
import numpy as np
from PIL import Image
from tqdm import tqdm
from collections import Counter

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Redirigir /tmp y HF cache a /workspace (disco real de 400 GB) ─────────────
#AQUI SE CAMBIAN RUTAS!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
os.makedirs("/workspace/tmp",       exist_ok=True)
os.makedirs("/workspace/hf_cache",  exist_ok=True)
os.makedirs("/workspace/model",     exist_ok=True)

os.environ["TMPDIR"]                = "/workspace/tmp"
os.environ["TEMP"]                  = "/workspace/tmp"
os.environ["TMP"]                   = "/workspace/tmp"
os.environ["HF_HOME"]               = "/workspace/hf_cache"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/workspace/hf_cache"
os.environ["HF_HUB_DISABLE_XET"]    = "1"

tempfile.tempdir = "/workspace/tmp"

# ── Paths ─────────────────────────────────────────────────────────────────────
EXTRACT_DIR  = "/workspace/exist2026"
DATASET_JSON = "/workspace/exist2026/archive/EXIST 2026 Dataset V0.1/EXIST 2026 Memes Dataset/training/EXIST2026_training.json"
MEMES_DIR    = "/workspace/exist2026/archive/EXIST 2026 Dataset V0.1/EXIST 2026 Memes Dataset/training/memes"
OUTPUT_DIR   = "/workspace/exist2026_qlora"
MODEL_LOCAL  = "/workspace/model"   # ← modelo descargado aquí sin doble copia

# Auto-detect JSON
if not os.path.exists(DATASET_JSON):
    candidates = []
    for root, _, files in os.walk(EXTRACT_DIR):
        candidates += [os.path.join(root, f) for f in files if f.endswith(".json")]
    if candidates:
        DATASET_JSON = candidates[0]
        print(f"[Auto-detect] JSON: {DATASET_JSON}")

# Auto-detect memes dir
if not os.path.exists(MEMES_DIR):
    for root, _, files in os.walk(EXTRACT_DIR):
        if os.path.basename(root) == "memes" and any(
            f.lower().endswith((".jpg", ".jpeg", ".png")) for f in files
        ):
            MEMES_DIR = root
            print(f"[Auto-detect] Memes: {MEMES_DIR}")
            break

assert os.path.exists(DATASET_JSON), f"JSON not found: {DATASET_JSON}"
assert os.path.exists(MEMES_DIR),    f"Memes dir not found: {MEMES_DIR}"
print(f"JSON        : {DATASET_JSON}")
print(f"Memes dir   : {MEMES_DIR}")
print(f"Memes count : {len(os.listdir(MEMES_DIR))}")

# ── Disk space ────────────────────────────────────────────────────────────────
import shutil
for path in ["/", "/workspace", "/workspace/model"]:
    try:
        total, used, free = shutil.disk_usage(path)
        print(f"  {path}: {free/1e9:.0f} GB libres / {total/1e9:.0f} GB total")
    except Exception:
        pass

# ── GPU info ──────────────────────────────────────────────────────────────────
N_GPUS = torch.cuda.device_count()
print(f"\nGPUs: {N_GPUS}")
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} — {p.total_memory / 1e9:.1f} GB")

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-VL-72B-Instruct"
# MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

# ── Labels ────────────────────────────────────────────────────────────────────
ALL_LABELS = [
    "IDEOLOGICAL-INEQUALITY",
    "STEREOTYPING-DOMINANCE",
    "OBJECTIFICATION",
    "SEXUAL-VIOLENCE",
    "MISOGYNY-NON-SEXUAL-VIOLENCE",
]

# ── Hyperparámetros — 2×A40 ───────────────────────────────────────────────────
THRESHOLD        = 0.5
MIN_ANNOTATORS   = 2
TRAIN_RATIO      = 0.70
VAL_RATIO        = 0.15
TEST_RATIO       = 0.15
BATCH_SIZE       = 2
GRAD_ACCUM_STEPS = 8
LEARNING_RATE    = 2e-5
NUM_EPOCHS       = 3
MAX_LENGTH       = 2048

print(f"\nModel           : {MODEL_ID}")
print(f"Model local dir : {MODEL_LOCAL}")
print(f"Effective batch : {BATCH_SIZE * GRAD_ACCUM_STEPS * N_GPUS}")
print(f"HF cache        : {os.environ['HF_HOME']}")
print(f"TMPDIR          : {os.environ['TMPDIR']}")

JSON        : /workspace/exist2026/archive/EXIST 2026 Dataset V0.1/EXIST 2026 Memes Dataset/training/EXIST2026_training.json
Memes dir   : /workspace/exist2026/archive/EXIST 2026 Dataset V0.1/EXIST 2026 Memes Dataset/training/memes
Memes count : 3984
  /: 271 GB libres / 429 GB total
  /workspace: 271 GB libres / 429 GB total
  /workspace/model: 271 GB libres / 429 GB total

GPUs: 2
  GPU 0: NVIDIA A40 — 47.7 GB
  GPU 1: NVIDIA A40 — 47.7 GB

Model           : Qwen/Qwen2.5-VL-72B-Instruct
Model local dir : /workspace/model
Effective batch : 32
HF cache        : /workspace/hf_cache
TMPDIR          : /workspace/tmp


In [ ]:
import subprocess

# Primero limpiar descargas anteriores
import shutil, os
for path in ["/workspace/hf_cache", "/workspace/tmp", "/root/.cache/huggingface"]:
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"Borrado: {path}")

#PRIMERA DESCARGA, SI AQUI SALE MAL NO TE PREOCUPES ABAJO HAY UN SEGUNDO INTENTO 

# Descargar directo a /workspace/model (sin doble copia)
print("Descargando modelo directo a /workspace/model ...")
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Qwen/Qwen2.5-VL-72B-Instruct",
    local_dir="/workspace/model",       # ← directo, sin cache intermedio
    ignore_patterns=["*.pt", "*.bin"],  # solo safetensors
)
print("Descarga completa")

In [ ]:
#DESCARGA INDIVIDUALMENTE LO QUE SALIÓ MAL Y REINTENTA HASTA QUE ESTÉ TODO EL MODELO
import os, time
from huggingface_hub import hf_hub_download, list_repo_files
#desca
REPO_ID   = "Qwen/Qwen2.5-VL-72B-Instruct"
LOCAL_DIR = "/workspace/model"
os.makedirs(LOCAL_DIR, exist_ok=True)

# Obtener lista de archivos (solo safetensors + config)
files = [
    f for f in list_repo_files(REPO_ID)
    if not f.endswith((".pt", ".bin"))
]
print(f"Archivos a descargar: {len(files)}")

# Descargar uno por uno con reintentos
for i, filename in enumerate(files):
    dest = os.path.join(LOCAL_DIR, filename)
    if os.path.exists(dest):
        print(f"[{i+1}/{len(files)}] Ya existe: {filename}")
        continue

    for intento in range(5):   # hasta 5 reintentos por archivo
        try:
            print(f"[{i+1}/{len(files)}] Descargando: {filename} (intento {intento+1})")
            hf_hub_download(
                repo_id=REPO_ID,
                filename=filename,
                local_dir=LOCAL_DIR,
            )
            print(f"  ✅ OK")
            break
        except Exception as e:
            print(f"  ❌ Error: {e}")
            if intento < 4:
                print(f"  Reintentando en 10s...")
                time.sleep(10)
            else:
                print(f"  ⚠️  Falló después de 5 intentos: {filename}")

print("\nDescarga completa.")

Archivos a descargar: 50
[1/50] Ya existe: .gitattributes
[2/50] Ya existe: LICENSE
[3/50] Ya existe: README.md
[4/50] Ya existe: chat_template.json
[5/50] Ya existe: config.json
[6/50] Ya existe: generation_config.json
[7/50] Ya existe: merges.txt
[8/50] Ya existe: model-00001-of-00038.safetensors
[9/50] Ya existe: model-00002-of-00038.safetensors
[10/50] Ya existe: model-00003-of-00038.safetensors
[11/50] Ya existe: model-00004-of-00038.safetensors
[12/50] Ya existe: model-00005-of-00038.safetensors
[13/50] Ya existe: model-00006-of-00038.safetensors
[14/50] Ya existe: model-00007-of-00038.safetensors
[15/50] Ya existe: model-00008-of-00038.safetensors
[16/50] Ya existe: model-00009-of-00038.safetensors
[17/50] Ya existe: model-00010-of-00038.safetensors
[18/50] Ya existe: model-00011-of-00038.safetensors
[19/50] Ya existe: model-00012-of-00038.safetensors
[20/50] Ya existe: model-00013-of-00038.safetensors
[21/50] Ya existe: model-00014-of-00038.safetensors
[22/50] Ya existe: model-

## 3. Load & Preprocess Dataset

In [4]:
def aggregate_labels(labels_task2_3: list, min_votes: int = 2) -> list:
    """
    Flatten per-annotator label lists and keep only labels confirmed
    by at least `min_votes` annotators. Ignores '-' entries.
    """
    flat = []
    for ann in labels_task2_3:
        if isinstance(ann, list):
            flat.extend(ann)
        elif isinstance(ann, str):
            flat.append(ann)

    counts = Counter(l for l in flat if l != "-" and l in ALL_LABELS)
    return sorted(l for l, c in counts.items() if c >= min_votes)


def load_dataset(json_path: str, memes_dir: str) -> list:
    with open(json_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    records = []
    for meme_id, item in tqdm(raw.items(), desc="Loading"):
        if "labels_task2_3" not in item:
            continue
        labels   = aggregate_labels(item["labels_task2_3"], MIN_ANNOTATORS)
        img_path = os.path.join(memes_dir, item["meme"])
        if not os.path.exists(img_path):
            print(f"  [WARN] Missing: {img_path}")
            continue
        records.append({
            "id":       meme_id,
            "text":     item["text"].strip(),
            "img_path": img_path,
            "labels":   labels,
        })

    print(f"\nLoaded {len(records)} valid records")
    return records


records  = load_dataset(DATASET_JSON, MEMES_DIR)
all_lbls = [l for r in records for l in r["labels"]]
print("\nLabel distribution:")
for lbl, cnt in Counter(all_lbls).most_common():
    print(f"  {lbl}: {cnt}")
print(f"  [no-sexism]: {sum(1 for r in records if not r['labels'])}")

Loading: 100%|██████████| 3984/3984 [00:00<00:00, 77583.22it/s]


Loaded 3984 valid records

Label distribution:
  STEREOTYPING-DOMINANCE: 1287
  OBJECTIFICATION: 1202
  IDEOLOGICAL-INEQUALITY: 1067
  SEXUAL-VIOLENCE: 563
  MISOGYNY-NON-SEXUAL-VIOLENCE: 448
  [no-sexism]: 1156


## 4. Stratified Split (70 / 15 / 15)

In [ ]:
# hace el split entre train, eval y test. El orden es el mismo que en el dataset original
#no lo cambies para que seed lo pueda replicar
mlb   = MultiLabelBinarizer(classes=ALL_LABELS)
Y     = mlb.fit_transform([r["labels"] for r in records])
X_idx = np.arange(len(records))

msss1 = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=(VAL_RATIO + TEST_RATIO), random_state=SEED
)
train_idx, valtest_idx = next(msss1.split(X_idx, Y))

msss2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    random_state=SEED,
)
val_sub, test_sub = next(msss2.split(valtest_idx, Y[valtest_idx]))
val_idx  = valtest_idx[val_sub]
test_idx = valtest_idx[test_sub]

train_data = [records[i] for i in train_idx]
val_data   = [records[i] for i in val_idx]
test_data  = [records[i] for i in test_idx]

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

Train: 2788 | Val: 598 | Test: 598


## 5. Instruction Tuning Format
### Prompt design
Three structured prompts, one per subtask. The notebook **trains on Prompt 3** (multi-label categories).  
Prompts 1 and 2 are included for zero-shot evaluation or cascade pipelines.

In [ ]:
# ── PROMPT 1 — Binary detection  (YES / NO) ───────────────────────────────────
SYSTEM_PROMPT_T1 = (
    "You are an expert in social bias detection.\n"
    "You must classify memes according to sexism.\n"
    "You must output ONLY valid JSON with key: label.\n"
    "No explanations or extra text."
)
USER_PROMPT_T1 = (
    "Task:\n"
    "Determine whether the meme contains sexism.\n\n"
    "Definition:\n"
    "A meme is sexist if it:\n"
    "- expresses sexist ideas or behaviors\n"
    "- depicts a sexist situation\n"
    "- refers to sexism, including criticizing it\n\n"
    "Important:\n"
    "- Even if the meme criticizes sexism, it is still YES\n"
    "- Consider sarcasm, irony, and implicit meaning\n"
    "- Use both image and text\n\n"
    "Decision:\n"
    "- If related to sexism → YES\n"
    "- If unrelated → NO\n\n"
    '{"label": "YES or NO"}'
)

# ── PROMPT 2 — Intention  (NO / DIRECT / JUDGEMENTAL) ─────────────────────────
SYSTEM_PROMPT_T2 = (
    "You are an expert in social bias and intention detection.\n"
    "You must classify memes according to their intention.\n"
    "You must output ONLY valid JSON with key: label.\n"
    "No explanations or extra text."
)
USER_PROMPT_T2 = (
    "Task:\n"
    "Classify the intention of the meme.\n\n"
    "Labels:\n"
    "- NO → not related to sexism\n"
    "- DIRECT → promotes or expresses sexism\n"
    "- JUDGEMENTAL → criticizes or condemns sexism\n\n"
    "Important:\n"
    "- DIRECT includes implicit or subtle sexism\n"
    "- JUDGEMENTAL includes irony, satire, or indirect criticism\n"
    "- Consider sarcasm and implicit meaning\n"
    "- Use both image and text\n\n"
    "Decision:\n"
    "- Unrelated → NO\n"
    "- Promotes sexism → DIRECT\n"
    "- Criticizes sexism → JUDGEMENTAL\n\n"
    '{"label": "NO | DIRECT | JUDGEMENTAL"}'
)

#ESTA ES LA QUE USA PARA ENTRENAR LAS DEMÁS SON DE PRUEBA
# ── PROMPT 3 — Multi-label con panel de jueces  ← TRAINING ───────────────────
SYSTEM_PROMPT_T3 = (
    "You are a panel of 5 diverse judges categorizing sexism in memes.\n"
    "Consider multiple cultural and personal perspectives.\n"
    "You must output ONLY valid JSON with key: labels.\n"
    "No explanations or extra text."
)
USER_PROMPT_T3 = (
    "Task:\n"
    "Assign all applicable sexism categories using 5 perspectives.\n\n"
    "Labels:\n"
    "- NO\n"
    "- IDEOLOGICAL-INEQUALITY\n"
    "- STEREOTYPING-DOMINANCE\n"
    "- OBJECTIFICATION\n"
    "- SEXUAL-VIOLENCE\n"
    "- MISOGYNY-NON-SEXUAL-VIOLENCE\n\n"
    "Definitions:\n"
    "- IDEOLOGICAL-INEQUALITY → rejects gender equality or attacks feminism\n"
    "- STEREOTYPING-DOMINANCE → assigns gender roles or male superiority\n"
    "- OBJECTIFICATION → reduces women to body or sexual attributes\n"
    "- SEXUAL-VIOLENCE → sexual coercion, harassment, or aggression\n"
    "- MISOGYNY-NON-SEXUAL-VIOLENCE → hate or non-sexual violence toward women\n\n"
    "JUDGE 1 (Teenager on TikTok): Would this get canceled? Which categories apply?\n"
    "JUDGE 2 (HR Manager): Which categories would trigger HR action?\n"
    "JUDGE 3 (Stand-up Comedian): Which stereotypes or tropes are being used?\n"
    "JUDGE 4 (Sociology Professor): Which systemic biases are present?\n"
    "JUDGE 5 (General Public): Which categories would most people identify?\n\n"
    "Important:\n"
    "- A category applies if 3 or more judges identify it\n"
    "- Multiple labels are allowed\n"
    '- If not related to sexism → return ["NO"]\n'
    "- Consider sarcasm and implicit meaning\n"
    "- Use both image and text\n\n"
    '{"labels": ["..."]}'
)


# ── Helpers ───────────────────────────────────────────────────────────────────

def _build_messages(system, user_text, img_path, assistant_response):
    return [
        {"role": "system",    "content": [{"type": "text",  "text": system}]},
        {"role": "user",      "content": [
            {"type": "image", "image": img_path},
            {"type": "text",  "text": user_text},
        ]},
        {"role": "assistant", "content": [{"type": "text",  "text": assistant_response}]},
    ]


def record_to_messages(record: dict) -> dict:
    output_labels      = record["labels"] if record["labels"] else ["NO"]
    assistant_response = json.dumps({"labels": output_labels})
    return {
        "id":       record["id"],
        "img_path": record["img_path"],
        "labels":   record["labels"],   # original (sin NO) — para evaluación
        "messages": _build_messages(
            SYSTEM_PROMPT_T3, USER_PROMPT_T3,
            record["img_path"], assistant_response,
        ),
    }


train_msgs = [record_to_messages(r) for r in train_data]
val_msgs   = [record_to_messages(r) for r in val_data]
test_msgs  = [record_to_messages(r) for r in test_data]

s = train_msgs[0]
print("Assistant:", s["messages"][-1]["content"][0]["text"])
print("Image ok :", os.path.exists(s["img_path"]))
print("Prompt   :", s["messages"][1]["content"][1]["text"][:80], "...")

Assistant: {"labels": ["MISOGYNY-NON-SEXUAL-VIOLENCE", "OBJECTIFICATION"]}
Image ok : True
Prompt   : Task:
Assign all applicable sexism categories using 5 perspectives.

Labels:
- N ...


## 6. Load Model & Processor with QLoRA

In [7]:
import subprocess, os, time

# ── Descargar modelo archivo por archivo con reintentos ───────────────────────
from huggingface_hub import hf_hub_download, list_repo_files

files = [f for f in list_repo_files(MODEL_ID) if not f.endswith((".pt", ".bin"))]
print(f"Archivos a descargar: {len(files)}")

for i, filename in enumerate(files):
    dest = os.path.join(MODEL_LOCAL, filename)
    if os.path.exists(dest):
        print(f"[{i+1}/{len(files)}] Ya existe: {filename}")
        continue
    for intento in range(5):
        try:
            print(f"[{i+1}/{len(files)}] {filename} (intento {intento+1})")
            hf_hub_download(repo_id=MODEL_ID, filename=filename, local_dir=MODEL_LOCAL)
            print(f"  ✅ OK")
            break
        except Exception as e:
            print(f"  ❌ {e}")
            if intento < 4:
                time.sleep(10)
            else:
                print(f"  ⚠️  Falló después de 5 intentos: {filename}")

print("\nDescarga completa.")

# ── Limpiar incompletos ───────────────────────────────────────────────────────
subprocess.run(["find", MODEL_LOCAL, "-name", "*.incomplete", "-delete"], capture_output=True)
subprocess.run(["find", MODEL_LOCAL, "-name", "*.lock",       "-delete"], capture_output=True)
print("Cache limpiado")

# ── 4-bit quantization ────────────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# ── Processor ─────────────────────────────────────────────────────────────────
processor = AutoProcessor.from_pretrained(
    MODEL_LOCAL,                  # ← desde disco local
    trust_remote_code=True,
    min_pixels=256 * 28 * 28,
    max_pixels=1280 * 28 * 28,
)

# ── Model — carga desde disco, sin descarga ───────────────────────────────────
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_LOCAL,                  # ← desde disco local
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)

model = prepare_model_for_kbit_training(model)

# ── LoRA ──────────────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


Archivos a descargar: 50
[1/50] Ya existe: .gitattributes
[2/50] Ya existe: LICENSE
[3/50] Ya existe: README.md
[4/50] Ya existe: chat_template.json
[5/50] Ya existe: config.json
[6/50] Ya existe: generation_config.json
[7/50] Ya existe: merges.txt
[8/50] Ya existe: model-00001-of-00038.safetensors
[9/50] Ya existe: model-00002-of-00038.safetensors
[10/50] Ya existe: model-00003-of-00038.safetensors
[11/50] Ya existe: model-00004-of-00038.safetensors
[12/50] Ya existe: model-00005-of-00038.safetensors
[13/50] Ya existe: model-00006-of-00038.safetensors
[14/50] Ya existe: model-00007-of-00038.safetensors
[15/50] Ya existe: model-00008-of-00038.safetensors
[16/50] Ya existe: model-00009-of-00038.safetensors
[17/50] Ya existe: model-00010-of-00038.safetensors
[18/50] Ya existe: model-00011-of-00038.safetensors
[19/50] Ya existe: model-00012-of-00038.safetensors
[20/50] Ya existe: model-00013-of-00038.safetensors
[21/50] Ya existe: model-00014-of-00038.safetensors
[22/50] Ya existe: model-

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/38 [00:00<?, ?it/s]

trainable params: 65,536,000 || all params: 73,476,313,344 || trainable%: 0.0892


## 7. Dataset & Collator

In [8]:
class MemeDataset(Dataset):
    """
    Tokenizes messages on-the-fly.
    Non-assistant tokens are masked with -100 so loss is computed
    only on the model's generated response.
    """
    def __init__(self, samples, processor, max_length=2048):
        self.samples    = samples
        self.processor  = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        text = self.processor.apply_chat_template(
            sample["messages"], tokenize=False, add_generation_prompt=False
        )
        image  = Image.open(sample["img_path"]).convert("RGB")
        inputs = self.processor(
            text=[text], images=[image],
            return_tensors="pt", truncation=True,
            max_length=self.max_length, padding=False,
        )
        input_ids      = inputs["input_ids"].squeeze(0)
        attention_mask = inputs["attention_mask"].squeeze(0)
        pixel_values   = inputs.get("pixel_values")
        labels         = input_ids.clone()

        # Mask up to and including the last <|im_start|>assistant token
        assist_ids = self.processor.tokenizer.encode(
            "<|im_start|>assistant", add_special_tokens=False
        )
        seq, mask_until = input_ids.tolist(), 0
        for i in range(len(seq) - len(assist_ids)):
            if seq[i : i + len(assist_ids)] == assist_ids:
                mask_until = i + len(assist_ids)
        labels[:mask_until] = -100

        out = {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}
        if pixel_values is not None:
            out["pixel_values"] = pixel_values.squeeze(0) if pixel_values.dim() == 4 else pixel_values
        if "image_grid_thw" in inputs:
            out["image_grid_thw"] = inputs["image_grid_thw"].squeeze(0)
        return out


def collate_fn(batch):
    pad_id  = processor.tokenizer.pad_token_id or 0
    max_len = max(b["input_ids"].size(0) for b in batch)
    ids, masks, lbls = [], [], []
    for b in batch:
        n = max_len - b["input_ids"].size(0)
        ids.append(torch.cat([b["input_ids"],       torch.full((n,), pad_id)]))
        masks.append(torch.cat([b["attention_mask"], torch.zeros(n, dtype=torch.long)]))
        lbls.append(torch.cat([b["labels"],          torch.full((n,), -100)]))
    out = {
        "input_ids":      torch.stack(ids),
        "attention_mask": torch.stack(masks),
        "labels":         torch.stack(lbls),
    }
    # Qwen2.5-VL: concatenar patches en dim=0, no hacer stack ni padding
    # image_grid_thw le dice al modelo cuántos patches pertenecen a cada imagen
    if "pixel_values" in batch[0]:
        out["pixel_values"] = torch.cat(
            [b["pixel_values"] for b in batch], dim=0
        )
    if "image_grid_thw" in batch[0]:
        out["image_grid_thw"] = torch.cat(
            [b["image_grid_thw"].unsqueeze(0) if b["image_grid_thw"].dim() == 1
             else b["image_grid_thw"] for b in batch], dim=0
        )
    return out


train_dataset = MemeDataset(train_msgs, processor, MAX_LENGTH)
val_dataset   = MemeDataset(val_msgs,   processor, MAX_LENGTH)
test_dataset  = MemeDataset(test_msgs,  processor, MAX_LENGTH)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
s0 = train_dataset[0]
print("input_ids:", s0["input_ids"].shape, "| labels:", s0["labels"].shape)

Train: 2788 | Val: 598 | Test: 598
input_ids: torch.Size([660]) | labels: torch.Size([660])


## 8. Training

In [ ]:
# El ultimo checkpoint esta aquí D:\GIL-Exist-2026\Fine tuning\Qwen2.5-VL-72B-Instruct\exist2026_qlora\checkpoint-250
#verifica las rutas

import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Detectar último checkpoint válido ─────────────────────────────────────────
checkpoints = sorted([
    d for d in os.listdir(OUTPUT_DIR)
    if d.startswith("checkpoint-")
], key=lambda x: int(x.split("-"+)[1]))

checkpoint_to_resume = None
for ckpt in reversed(checkpoints):
    ckpt_path = os.path.join(OUTPUT_DIR, ckpt)
    adapter_file = os.path.join(ckpt_path, "adapter_model.safetensors")
    if os.path.exists(adapter_file):
        try:
            from safetensors import safe_open
            with safe_open(adapter_file, framework="pt", device="cpu") as f:
                _ = f.keys()
            checkpoint_to_resume = ckpt_path
            print(f"✅ Checkpoint válido encontrado: {ckpt}")
            break
        except Exception as e:
            print(f"⚠️  Checkpoint corrupto, saltando: {ckpt} — {e}")

if checkpoint_to_resume is None:
    print("🆕 No hay checkpoints válidos — entrenando desde cero")

# ── Training args ─────────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-5,
    weight_decay=0.0,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    max_grad_norm=1.0,
    optim="adamw_torch_fused",
    logging_strategy="steps",
    logging_steps=5,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    report_to="none",
    disable_tqdm=False,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    processing_class=processor.tokenizer,
)

# ── Entrenar ──────────────────────────────────────────────────────────────────
trainer.train(resume_from_checkpoint=checkpoint_to_resume)
trainer.save_model(os.path.join(OUTPUT_DIR, "best_model"))
print("Training complete. Model saved.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


✅ Checkpoint válido encontrado: checkpoint-200


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
Casting fp32 inputs back to torch.bfloat16 for flash-attn compatibility.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss,Validation Loss
250,0.107700,0.110416


/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


In [ ]:
#si se llena el gpu usa este comando para limpiarlo y retoma desde el ultimo checkpoint
import torch, gc

# 1. Liberar cache de CUDA
torch.cuda.empty_cache()

# 2. Forzar garbage collector de Python
gc.collect()

# 3. Ver cuánto quedó libre
for i in range(torch.cuda.device_count()):
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    used  = torch.cuda.memory_allocated(i) / 1e9
    print(f"GPU {i}: {used:.1f} GB usados / {total:.1f} GB total")

GPU 0: 20.4 GB usados / 47.7 GB total
GPU 1: 27.2 GB usados / 47.7 GB total


## 9. Evaluation

In [ ]:
# a paritr de aqui, no esta probado pero es para evaluar el modelo en el set de test, si quieres probarlo hazlo con cuidado y verifica las rutas de los modelos y los datos
def parse_labels(decoded: str) -> list:
    """Parse {labels:[...]} or bare [...] from model output. Drops 'NO'."""
    try:
        m = re.search(r"\{.*?\}", decoded, re.DOTALL)
        raw = json.loads(m.group()).get("labels", []) if m else []
        if not raw:
            m2  = re.search(r"\[.*?\]", decoded, re.DOTALL)
            raw = json.loads(m2.group()) if m2 else []
        return [l for l in raw if l in ALL_LABELS]
    except Exception:
        return []


def generate_predictions(model, processor, samples, max_new_tokens=128):
    model.eval()
    preds = []
    with torch.no_grad():
        for s in tqdm(samples, desc="Evaluating"):
            text = processor.apply_chat_template(
                s["messages"][:-1], tokenize=False, add_generation_prompt=True
            )
            image  = Image.open(s["img_path"]).convert("RGB")
            inputs = processor(
                text=[text], images=[image], return_tensors="pt",
                truncation=True, max_length=MAX_LENGTH,
            ).to(model.device)

            out_ids = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, temperature=None, top_p=None,
            )
            n = inputs["input_ids"].shape[-1]
            decoded = processor.tokenizer.decode(out_ids[0][n:], skip_special_tokens=True).strip()
            preds.append(parse_labels(decoded))
    return preds


def evaluate_multilabel(true_labels, pred_labels):
    Y_true = mlb.transform(true_labels)
    Y_pred = mlb.transform(pred_labels)
    results = {
        "f1_macro":    f1_score(Y_true, Y_pred, average="macro",    zero_division=0),
        "f1_micro":    f1_score(Y_true, Y_pred, average="micro",    zero_division=0),
        "f1_weighted": f1_score(Y_true, Y_pred, average="weighted", zero_division=0),
    }
    for lbl, sc in zip(ALL_LABELS, f1_score(Y_true, Y_pred, average=None, zero_division=0)):
        results[f"f1_{lbl}"] = sc
    return results


test_preds = generate_predictions(model, processor, test_msgs)
test_true  = [r["labels"] for r in test_msgs]
metrics    = evaluate_multilabel(test_true, test_preds)

print("\n" + "="*52)
print("TEST SET EVALUATION")
print("="*52)
print(f"  F1 Macro   : {metrics['f1_macro']:.4f}")
print(f"  F1 Micro   : {metrics['f1_micro']:.4f}")
print(f"  F1 Weighted: {metrics['f1_weighted']:.4f}")
print("\nPer-class F1:")
for lbl in ALL_LABELS:
    print(f"  {lbl:<35}: {metrics[f'f1_{lbl}']:.4f}")

## 10. Inference Example

In [ ]:
def predict_single(model, processor, img_path, max_new_tokens=64):
    """
    Classifica un meme individual con Prompt 3.
    El modelo lee el texto directamente de la imagen.
    """
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT_T3}]},
        {"role": "user",   "content": [
            {"type": "image", "image": img_path},
            {"type": "text",  "text": USER_PROMPT_T3},
        ]},
    ]
    text  = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image = Image.open(img_path).convert("RGB")
    inputs = processor(
        text=[text], images=[image], return_tensors="pt",
        truncation=True, max_length=MAX_LENGTH,
    ).to(model.device)

    model.eval()
    with torch.no_grad():
        out_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, temperature=None, top_p=None,
        )

    n = inputs["input_ids"].shape[-1]
    decoded = processor.tokenizer.decode(out_ids[0][n:], skip_special_tokens=True).strip()
    pred    = parse_labels(decoded)
    print(f"Raw output   : {decoded}")
    print(f"Parsed labels: {pred}")
    return pred


demo = test_msgs[0]
print(f"Meme text  : {test_data[0]['text']}")
print(f"True labels: {demo['labels']}\n")
predicted = predict_single(model, processor, img_path=demo["img_path"])

## 11. Save & Export
Merge LoRA adapters into base weights for deployment.

In [ ]:
MERGED_DIR = os.path.join(OUTPUT_DIR, "merged_model")
os.makedirs(MERGED_DIR, exist_ok=True)

merged_model = model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
processor.save_pretrained(MERGED_DIR)

print(f"Merged model saved to: {MERGED_DIR}")